# Домашняя работа 9: Object Tracking

В этом домашнем задании мы будем работать с задачей отслеживания объектов (object tracking) на датасете MOT15. 

План домашнего задания:
1. Реализация класса Dataset для работы с MOT15
2. Детекция объектов с помощью YOLOv8
3. Трекинг объектов с использованием DeepSORT
4. Визуализация результатов трекинга
5. Подсчет объектов, пересекающих заданную линию

Импортируем необходимые библиотеки:

In [ ]:
# https://www.kaggle.com/datasets/andreykurdyubov/cv-hw9-yolo-detections
# Датасет с сохраненными детекциями и треками из этого задания

In [1]:
# Установим Ultralytics и DeepSORT
!pip install ultralytics deep-sort-realtime

# # Скачаем данные
# !wget https://motchallenge.net/data/MOT15.zip -O MOT15.zip
# !unzip -q MOT15.zip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 74.8 MB/s eta 0:00:00:00:010:01


In [2]:
import os
import json
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional
from collections import defaultdict

import torch
from torch.utils.data import Dataset
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import matplotlib.pyplot as plt

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
DeepSort??

## Часть 1: Работа с датасетом MOT15

В первой части мы реализуем класс для работы с датасетом MOT15. Этот датасет содержит последовательности изображений с размеченными объектами для задачи трекинга.

Реализуйте класс `MOT15Dataset`, который должен:
1. Принимать путь к директории с данными и название последовательности (sequence)
2. Загружать список изображений для данной последовательности
3. Предоставлять метод для получения изображения по индексу
4. Загружать ground truth разметку, если она доступна

In [ ]:
data_root = "/kaggle/input/datasets/mdraselsarker/mot15-challenge-dataset/MOT15/train"
sequence = "ADL-Rundle-6"
folder = Path(data_root) / sequence
folder.stem, folder

In [ ]:
images = folder / "img1"
data_list = sorted(images.rglob("*.jpg"))

In [ ]:
idx = 524
frame = data_list[idx].stem
frame = int(frame.split(".")[0])
frame

In [ ]:
gt = folder / "gt" / 'gt.txt'
gt_df = pd.read_csv(gt, names=["frame", "id", "x", "y", "w", "h", "score", "3dx", "3dy", "3dz"])
gt_df.head()

In [ ]:
gt = folder / "gt" / 'gt1.txt'

gt.is_file()

In [ ]:
x = gt_df[gt_df["frame"]==1000][["x", "y", "w", "h", "id"]].to_numpy()
x

In [ ]:
# BEGIN STUDENT
class MOT15Dataset(Dataset):
    def __init__(self, data_root: str, sequence: str):
        """
        Args:
            data_root: путь к корневой директории датасета
            sequence: название последовательности (например, 'TUD-Stadtmitte')
        """
        folder = Path(data_root) / sequence
        images = folder / "img1"
        self.data_list = sorted(images.rglob("*.jpg"))
        
        gt = folder / "gt" / 'gt.txt'

        if gt.is_file():
            self.gt_df = pd.read_csv(gt, names=["frame", "id", "x", "y", "w", "h", "score", "3dx", "3dy", "3dz"])
        else:
            self.gt_df = None

    def __len__(self) -> int:
        return len(self.data_list)

    def __getitem__(self, idx: int) -> tuple[np.ndarray, Optional[np.ndarray]]:
        """
        Returns:
            tuple: (изображение, ground truth разметка если доступна)
        """
        frame = idx + 1
        img = cv2.imread(self.data_list[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        

        if self.gt_df is not None:
            boxes = self.gt_df[self.gt_df["frame"]==frame][["x", "y", "w", "h", "id"]].to_numpy()
            return img, boxes
        return img
                        
        
# END STUDENT

In [4]:
dataset = MOT15Dataset("/kaggle/input/datasets/mdraselsarker/mot15-challenge-dataset/MOT15/train", "ADL-Rundle-6")
print(f"Dataset length: {len(dataset)}")
    
img, gt = dataset[0]
print(f"Image shape: {img.shape}")
if gt is not None:
    print(f"Ground truth shape: {gt.shape}")
else:
    print("No ground truth available")
        
assert isinstance(img, np.ndarray), "Image should be a numpy array"
assert len(img.shape) == 3, "Image should have 3 dimensions"
assert img.shape[2] == 3, "Image should have 3 channels"

Dataset length: 525
Image shape: (1080, 1920, 3)
Ground truth shape: (4, 5)


Загрузите код в LMS.

In [3]:
#решение автора
class MOT15Dataset(Dataset):
    def __init__(self, data_root: str, sequence: str):
        self.data_root = Path(data_root)
        self.sequence = sequence
        self.sequence_path = self.data_root / sequence
        
        # Загрузка списка изображений
        self.img_dir = self.sequence_path / 'img1'
        self.image_files = sorted(list(self.img_dir.glob('*.jpg')))
        
        # Загрузка ground truth, если есть
        self.gt_file = self.sequence_path / 'gt' / 'gt.txt'
        self.gt_data = None
        if self.gt_file.exists():
            self.gt_data = pd.read_csv(self.gt_file, header=None)
            self.gt_data.columns = ['frame', 'id', 'bb_left', 'bb_top', 'bb_width', 'bb_height', 'conf', 'x', 'y', 'z']

    def __len__(self) -> int:
        return len(self.image_files)

    def __getitem__(self, idx: int) -> tuple[np.ndarray, Optional[np.ndarray]]:
        # Загрузка изображения
        img_path = self.image_files[idx]
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Получение ground truth для текущего кадра
        gt = None
        if self.gt_data is not None:
            frame_data = self.gt_data[self.gt_data['frame'] == idx + 1]
            if not frame_data.empty:
                gt = frame_data[['bb_left', 'bb_top', 'bb_width', 'bb_height', 'id']].values
        
        return image, gt

## Часть 2: Детекция объектов с YOLOv8

В этой части мы будем использовать предобученную модель YOLOv8-M для детекции объектов на кадрах видео. 
Результаты детекции нужно сохранить в формате, подходящем для дальнейшего трекинга.

Формат файла с детекциями:
- Cписок словарей, где каждый словарь содержит:
  - 'frame': номер кадра (начиная с 1)
  - 'bb_left': x-координата левого верхнего угла ограничивающей рамки
  - 'bb_top': y-координата левого верхнего угла ограничивающей рамки
  - 'bb_width': ширина ограничивающей рамки
  - 'bb_height': высота ограничивающей рамки
  - 'conf': уверенность детекции (от 0 до 1)

Пример записи в JSON:
{
    "frame": 1,
    "bb_left": 100.5,
    "bb_top": 150.2,
    "bb_width": 50.0,
    "bb_height": 100.0,
    "conf": 0.95
}

Важно:
1. Сохраняются только детекции людей
2. Все координаты и размеры указываются в пикселях, в формате float
3. Номера кадров начинаются с 1
4. Уверенность детекции (conf) должна быть в диапазоне [0, 1]

In [ ]:
bb_left.item()

In [ ]:
import matplotlib.patches as patches

model = YOLO('yolov8m.pt')
frame = 0
image, gt = dataset[frame]
fig, ax = plt.subplots(figsize=(10, 6))

ax.imshow(image)
result = model.predict(source=image)
cls = result[0].boxes.cls
mask = cls == 0

conf = result[0].boxes.conf
tensors = result[0].boxes.xyxy
conf = conf[mask]
tensors = tensors[mask]
print(len(tensors))

boxes = []

for k in range(len(tensors)):
    bb_left, bb_top, x2, y2 = tensors[k]
    bb_width, bb_height = x2 - bb_left, y2 - bb_top
    xmin, ymin = (x - width_pix/2), (y - height_pix/2)
    rect = patches.Rectangle((bb_left, bb_top), bb_width, bb_height, linewidth=2, edgecolor='r', facecolor='none')
    ax.add_patch(rect)
    ax.text(bb_left, bb_top, f'{conf[k]:.2f}', fontsize=12, color='white', ha='left', va='bottom')
    boxes.append({
        "frame": frame,
        "bb_left": bb_left.item(),
        "bb_top": bb_top.item(),
        "bb_width": bb_width.item(),
        "bb_height": bb_height.item(),
        "conf": conf[k].item()
        }
    )

with open("out.json", "w") as file:
    json.dump(boxes, file)

In [ ]:
# BEGIN STUDENT
def detect_objects(dataset: MOT15Dataset, output_path: str):
    """
    Детекция объектов на последовательности изображений
    Args:
        dataset: датасет с изображениями
        output_path: путь для сохранения результатов
    """
    model = YOLO('yolov8m.pt')
    boxes = []
    
    for frame in range(len(dataset)):
        image, gt = dataset[frame]
        result = model.predict(source=image)
        cls = result[0].boxes.cls
        mask = cls == 0
        
        conf = result[0].boxes.conf
        tensors = result[0].boxes.xyxy
        conf = conf[mask]
        tensors = tensors[mask]
               
        for k in range(len(tensors)):
            bb_left, bb_top, x2, y2 = tensors[k]
            bb_width, bb_height = x2 - bb_left, y2 - bb_top

            boxes.append({
                "frame": frame + 1,
                "bb_left": bb_left.item(),
                "bb_top": bb_top.item(),
                "bb_width": bb_width.item(),
                "bb_height": bb_height.item(),
                "conf": conf[k].item()
                }
            )
        
    with open(output_path, "w") as file:
        json.dump(boxes, file)


# END STUDENT

In [6]:
# Создаем датасет для последовательности ADL-Rundle-6
dataset = MOT15Dataset("/kaggle/input/datasets/mdraselsarker/mot15-challenge-dataset/MOT15/train", "ADL-Rundle-6")

# Путь для сохранения результатов
output_path = "yolo_detections.json"

# Запускаем детекцию объектов
detect_objects(dataset, output_path)

# Проверяем результаты
with open(output_path, 'r') as f:
    detections = json.load(f)

print(f"Всего найдено {len(detections)} детекций")
print("\nПример первой детекции:")
print(json.dumps(detections[0], indent=2, ensure_ascii=False))

Всего найдено 5325 детекций

Пример первой детекции:
{
  "frame": 1,
  "bb_left": 1704.617431640625,
  "bb_top": 388.6498718261719,
  "bb_width": 161.8121337890625,
  "bb_height": 339.8596496582031,
  "conf": 0.8706992268562317
}


Полученный json залейте в LMS.

In [5]:
# решение автора
def detect_objects(dataset: MOT15Dataset, output_path: str):
    # Загрузка модели YOLOv8
    model = YOLO('yolov8m.pt')
    
    detections = []
    
    # Детекция объектов на каждом кадре
    for frame_idx in range(len(dataset)):
        image, _ = dataset[frame_idx]
        results = model(image, verbose=False)
        
        # Сохраняем только детекции людей (class_id = 0 в COCO)
        for r in results:
            boxes = r.boxes
            for box_idx in range(len(boxes)):
                if boxes.cls[box_idx].item() == 0:  # class_id для человека
                    box = boxes.xyxy[box_idx].cpu().numpy()
                    conf = boxes.conf[box_idx].item()
                    
                    detections.append({
                        'frame': frame_idx + 1,
                        'bb_left': float(box[0]),
                        'bb_top': float(box[1]),
                        'bb_width': float(box[2] - box[0]),
                        'bb_height': float(box[3] - box[1]),
                        'conf': float(conf)
                    })
    
    # Сохранение результатов
    if output_path.endswith('.json'):
        with open(output_path, 'w') as f:
            json.dump(detections, f)

## Часть 3: Трекинг с DeepSORT

В этой части мы используем алгоритм DeepSORT для трекинга объектов на основе полученных детекций YOLOv8. 
DeepSORT не только использует информацию о положении объектов, но и извлекает визуальные признаки для более надежного отслеживания.

Формат файла с результатами трекинга:
- Список словарей, где каждый словарь содержит:
  - 'frame': номер кадра (начиная с 1)
  - 'id': уникальный идентификатор трека (целое число)
  - 'bb_left': x-координата левого верхнего угла ограничивающей рамки
  - 'bb_top': y-координата левого верхнего угла ограничивающей рамки
  - 'bb_width': ширина ограничивающей рамки
  - 'bb_height': высота ограничивающей рамки

Пример записи в JSON:
{
    "frame": 1,
    "id": 5,
    "bb_left": 100.5,
    "bb_top": 150.2,
    "bb_width": 50.0,
    "bb_height": 100.0
}

Важно:
1. ID трека является уникальным для каждого отслеживаемого объекта и сохраняется между кадрами
2. В выходной файл попадают только подтвержденные треки (is_confirmed() == True)
3. Все координаты и размеры указываются в пикселях, в формате float
4. Номера кадров начинаются с 1
5. Один и тот же объект (ID) может появляться в нескольких последовательных кадрах

In [ ]:
from collections import defaultdict

detections_path = "/kaggle/input/datasets/andreykurdyubov/cv-hw9-yolo-detections/yolo_detections (1).json"

with open(detections_path, 'r') as f:
    detections = json.load(f)

detects = defaultdict(list)

for detection in detections:
    frame = detection["frame"]
    bbox = [detection["bb_left"], detection["bb_top"], detection["bb_width"], detection["bb_height"]]
    conf = detection["conf"]
    detects[frame].append((bbox, conf, "person"))


In [8]:
# BEGIN STUDENT
def track_objects(dataset: MOT15Dataset, detections_path: str, output_path: str):
    """
    Трекинг объектов с использованием DeepSORT
    Args:
        dataset: датасет с изображениями
        detections_path: путь к файлу с детекциями
        output_path: путь для сохранения результатов трекинга
    """
    tracker = DeepSort(
        max_age=30,             # сколько кадров помнить пропавший объект
        n_init=3,               # сколько кадров подряд нужно фиксировать объект для старта трека
        nms_max_overlap=1.0,    # порог Non-Maximum Suppression отключаем
        embedder="mobilenet"    # встроенная нейросеть для извлечения эмбеддингов для ReID
        )
    
    with open(detections_path, 'r') as f:
        detections = json.load(f)

    detects_by_frame = defaultdict(list)

    # преобразуем в формат для треккера
    for detection in detections:
        frame = detection["frame"]
        bbox = [detection["bb_left"], detection["bb_top"], detection["bb_width"], detection["bb_height"]]
        conf = detection["conf"]
        detects_by_frame[frame].append((bbox, conf, "person"))

    track_list = []
    for k in range(len(dataset)):
        frame, _ = dataset[k]
        tracks = tracker.update_tracks(detects_by_frame[k+1], frame=frame)

        for track in tracks:
            # Пропускаем временные/неподтвержденные треки
            if not track.is_confirmed():
                continue
                
            track_id = track.track_id
            
            # Получаем [left, top, right, bottom]
            left, top, right, bottom = track.to_ltrb()
            
            # Переводим обратно в формат [left, top, width, height]
            width = right - left
            height = bottom - top
            
            # Создаем словарь в точности по вашему шаблону
            record = {
                "frame": k + 1,
                "id": int(track_id),
                "bb_left": left,
                "bb_top": top,
                "bb_width": width,
                "bb_height": height
            }
            
            track_list.append(record)
            
    with open(output_path, "w") as file:
        json.dump(track_list, file)
# END STUDENT

In [9]:
# Создаем датасет для последовательности ADL-Rundle-6
dataset = MOT15Dataset("/kaggle/input/datasets/mdraselsarker/mot15-challenge-dataset/MOT15/train", "ADL-Rundle-6")

# Пути к файлам с детекциями и для сохранения результатов
detections_path = "/kaggle/input/datasets/andreykurdyubov/cv-hw9-yolo-detections/yolo_detections (1).json"
output_path = "tracking_results_2.json"

# Запускаем трекинг объектов
track_objects(dataset, detections_path, output_path)

# Проверяем результаты
with open(output_path, 'r') as f:
    tracks = json.load(f)

# Анализируем результаты
unique_tracks = len(set(t['id'] for t in tracks))
total_detections = len(tracks)
frames_count = max(t['frame'] for t in tracks)

print(f"Статистика трекинга:")
print(f"Всего уникальных треков: {unique_tracks}")
print(f"Всего детекций в треках: {total_detections}")
print(f"Количество обработанных кадров: {frames_count}")
print("\nПример первой записи трека:")
print(json.dumps(tracks[0], indent=2, ensure_ascii=False))

# Распределение треков по кадрам
tracks_per_frame = {}
for track in tracks:
    frame = track['frame']
    if frame not in tracks_per_frame:
        tracks_per_frame[frame] = 0
    tracks_per_frame[frame] += 1

print(f"\nСреднее количество треков на кадр: {np.mean(list(tracks_per_frame.values())):.2f}")

/usr/local/lib/python3.12/dist-packages/deep_sort_realtime/embedder/embedder_pytorch.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Статистика трекинга:
Всего уникальных треков: 63
Всего детекций в треках: 7881
Количество обработанных кадров: 525

Пример первой записи трека:
{
  "frame": 3,
  "id": 1,
  "bb_left": 1688.8188813715701,
  "bb_top": 388.18486075669614,
  "bb_width": 162.193404583943,
  "bb_height": 340.65309375923005
}

Среднее количество треков на кадр: 15.07


In [ ]:
# решение автора
def track_objects(dataset: MOT15Dataset, detections_path: str, output_path: str):
    # Инициализация DeepSORT
    tracker = DeepSort(
        max_age=30,
        n_init=3,
        nms_max_overlap=1.0,
        max_cosine_distance=0.3,
        nn_budget=None,
        override_track_class=None,
        embedder="mobilenet",
        half=True,
        bgr=False,
        embedder_gpu=True
    )
    
    # Загрузка детекций
    if detections_path.endswith('.json'):
        with open(detections_path, 'r') as f:
            detections = json.load(f)
    else:
        detections = pd.read_csv(detections_path).to_dict('records')
    
    tracks = []
    frame_detections = {}
    
    # Группировка детекций по кадрам
    for det in detections:
        frame = det['frame']
        if frame not in frame_detections:
            frame_detections[frame] = []
        frame_detections[frame].append(det)
    
    # Применение трекинга
    for frame_idx in range(len(dataset)):
        image, _ = dataset[frame_idx]
        frame_dets = frame_detections.get(frame_idx + 1, [])
        
        if frame_dets:
            # Подготовка детекций для DeepSORT
            detection_list = []
            for det in frame_dets:
                x1 = det['bb_left']
                y1 = det['bb_top']
                w = det['bb_width']
                h = det['bb_height']
                conf = det['conf']
                
                detection_list.append(([x1, y1, w, h], conf, 'person'))
            
            # Обновление треков
            track_results = tracker.update_tracks(detection_list, frame=image)

            # Сохранение результатов трекинга
            for track in track_results:
                if not track.is_confirmed():
                    continue
                    
                ltrb = track.to_ltrb()
                tracks.append({
                    'frame': frame_idx + 1,
                    'id': int(track.track_id),
                    'bb_left': float(ltrb[0]),
                    'bb_top': float(ltrb[1]),
                    'bb_width': float(ltrb[2] - ltrb[0]),
                    'bb_height': float(ltrb[3] - ltrb[1])
                })
    
    # Сохранение результатов
    if output_path.endswith('.json'):
        with open(output_path, 'w') as f:
            json.dump(tracks, f)

Полученный json залейте в LMS.

## Часть 4: Визуализация результатов

Реализуйте функцию для визуализации результатов трекинга. Функция должна возвращать изображение с отрисованными bounding box'ами и ID объектов.

In [ ]:
# BEGIN STUDENT
def visualize_tracking(image: np.ndarray, tracks: list[dict]) -> np.ndarray:
    """
    Визуализация результатов трекинга на изображении
    Args:
        image: исходное изображение
        tracks: список треков для текущего кадра
    Returns:
        np.ndarray: изображение с визуализацией
    """
    for track in tracks:
        track_id = track["id"]
        bb_left, bb_top, bb_width, bb_height = int(track["bb_left"]), int(track["bb_top"]), int(track["bb_width"]), int(track["bb_height"])
        cv2.rectangle(image, (bb_left, bb_top), (bb_left + bb_width, bb_top + bb_height), (255, 0, 0), 2)
        cv2.putText(image, str(track_id), (bb_left, bb_top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 3)

    return image
# END STUDENT

In [ ]:
# Создаем датасет для последовательности ADL-Rundle-6
dataset = MOT15Dataset("/kaggle/input/datasets/mdraselsarker/mot15-challenge-dataset/MOT15/train", "ADL-Rundle-6")

# Загружаем результаты детекций и трекинга
with open("/kaggle/input/datasets/andreykurdyubov/cv-hw9-yolo-detections/yolo_detections (1).json", 'r') as f:
    detections = json.load(f)
with open("/kaggle/input/datasets/andreykurdyubov/cv-hw9-yolo-detections/tracking_results_2.json", 'r') as f:
    tracks = json.load(f)

# Выбираем кадр для визуализации (например, 100-й кадр)
frame_idx = 2
image, _ = dataset[frame_idx]

# Получаем детекции и треки для выбранного кадра
#frame_detections = [d for d in detections if d['frame'] == frame_idx + 1]
frame_tracks = [t for t in tracks if t['frame'] == frame_idx + 1]

# Создаем фигуру с двумя изображениями
plt.figure(figsize=(20, 10))

# Визуализируем результаты трекинга
track_vis = visualize_tracking(image, frame_tracks)
plt.imshow(track_vis)
plt.title(f'Результаты трекинга DeepSORT (кадр {frame_idx + 1})\nВсего треков: {len(frame_tracks)}')
plt.axis('off')

plt.tight_layout()
plt.show()

Код функции загрузите в LMS.

In [ ]:
# решение автора
def visualize_tracking(image: np.ndarray, tracks: list[dict]) -> np.ndarray:
    # Создаём копию изображения для визуализации
    vis_image = image.copy()
    
    # Отрисовка каждого трека
    for track in tracks:
        # Получаем координаты bounding box'а
        x1 = int(track['bb_left'])
        y1 = int(track['bb_top'])
        x2 = int(x1 + track['bb_width'])
        y2 = int(y1 + track['bb_height'])
        track_id = track['id']
        
        # Генерируем уникальный цвет для каждого ID
        color = tuple(map(int, np.random.randint(0, 255, 3)))
        
        # Рисуем bounding box
        cv2.rectangle(vis_image, (x1, y1), (x2, y2), color, 2)
        
        # Добавляем текст с ID
        text = f'ID: {track_id}'
        cv2.putText(vis_image, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        
    return vis_image

## Часть 5: Подсчет пересечений

В этой части мы реализуем функционал подсчета количества людей, пересекающих заданную линию на видео. Это важная задача для различных приложений, таких как анализ потока посетителей в торговых центрах, подсчет пассажиров на станциях или мониторинг движения пешеходов на улицах.

Входные данные:
1. Результаты трекинга в формате JSON, где каждая запись содержит:
   - 'frame': номер кадра (начиная с 1)
   - 'id': уникальный идентификатор трека
   - 'bb_left': x-координата левого верхнего угла
   - 'bb_top': y-координата левого верхнего угла
   - 'bb_width': ширина bbox'а
   - 'bb_height': высота bbox'а

2. Линия подсчета, заданная двумя точками:
   - Начальная точка (x1, y1)
   - Конечная точка (x2, y2)

Алгоритм работы:
1. Для каждого уникального трека (ID):
   - Отслеживаем последовательные позиции центра bbox'а
   - Проверяем пересечение траектории движения с заданной линией
   - Учитываем только первое пересечение линии каждым объектом

2. Определение пересечения:
   - Рассматриваем движение объекта как отрезок между последовательными положениями
   - Используем алгоритм пересечения двух отрезков
   - Учитываем направление движения для корректного подсчета

Выходные данные - количество уникальных пересечений линии.

In [5]:
with open("/kaggle/input/datasets/andreykurdyubov/cv-hw9-yolo-detections/tracking_results_2.json", 'r') as f:
    tracks = json.load(f)

tracks[:2]

[{'frame': 3,
  'id': 1,
  'bb_left': 1688.8188813715701,
  'bb_top': 388.18486075669614,
  'bb_width': 162.193404583943,
  'bb_height': 340.65309375923005},
 {'frame': 3,
  'id': 2,
  'bb_left': 263.79816108622975,
  'bb_top': 457.37094674115826,
  'bb_width': 93.98978809133399,
  'bb_height': 249.26977342392956}]

In [ ]:
frames_id[3]

In [ ]:
def check_cross(x1, y1, x2, y2, x_l1, y_l1, x_l2, y_l2):
    """
    проверка, что отрезок (x1, y1) - (x2, y2) пересекает (x_l1, y_l1) - (x_l2, y_l2)
    """
    # проверяем что векторное произведение меняет знак z-компоненты
    z1 = (x2-x1)*(y_l1-y1) - (y2-y1)*(x_l1-x1)
    z2 = (x2-x1)*(y_l2-y1) - (y2-y1)*(x_l2-x1)

    z_l1 = (x_l2-x_l1)*(y1-y_l1) - (y_l2-y_l1)*(x1-x_l1)
    z_l2 = (x_l2-x_l1)*(y2-y_l2) - (y_l2-y_l1)*(x2-x_l2)

    return (z1*z2 < 0) and (z_l1*z_l2 < 0)

In [ ]:
x1, y1, x2, y2, x_l1, y_l1, x_l2, y_l2 = 1, 4, 4, 3, 3, 5, 3, 2
check_cross(x1, y1, x2, y2, x_l1, y_l1, x_l2, y_l2)

In [7]:
def check_cross(x1, y1, x2, y2, x_l1, y_l1, x_l2, y_l2):
    """
    проверка, что отрезок (x1, y1) - (x2, y2) пересекает (x_l1, y_l1) - (x_l2, y_l2)
    """
    # проверяем что векторное произведение меняет знак z-компоненты
    z1 = (x2-x1)*(y_l1-y1) - (y2-y1)*(x_l1-x1)
    z2 = (x2-x1)*(y_l2-y1) - (y2-y1)*(x_l2-x1)

    z_l1 = (x_l2-x_l1)*(y1-y_l1) - (y_l2-y_l1)*(x1-x_l1)
    z_l2 = (x_l2-x_l1)*(y2-y_l2) - (y_l2-y_l1)*(x2-x_l2)

    return (z1*z2 < 0) and (z_l1*z_l2 < 0)

frames_id = defaultdict(dict)
all_frames = set()

# проход по трекам, собираем историю
for track in tracks:
    frame = track["frame"]
    all_frames.add(frame)
    id_frame = track["id"]
    x_center = track["bb_left"] + track["bb_width"]/2
    y_center = track["bb_top"] + track["bb_height"]/2
    frames_id[frame][id_frame] = (x_center, y_center)

# frames_id[3]
# '''
{1: (1769.9155836635416, 558.5114076363111),
 2: (310.7930551318967, 582.005833453123),
 3: (59.51568041253265, 610.7293407858306),
 4: (1288.0091620736293, 598.9503381744635),
 5: (1327.99111133226, 556.2222750710608),
 6: (1900.6403542980013, 478.5830614966998),
 7: (162.6133789067276, 621.3300300706703),
 8: (84.93331093362578, 646.8613059212744)}
# '''

all_frames = sorted(list(all_frames))
id_centers = defaultdict(bool)
crossings = 0
id_crossed = defaultdict(bool)

line_start = (960, 0)    # Верхняя точка линии (x=960, y=0)
line_end = (960, 1080)   # Нижняя точка линии (x=960, y=1080)
x_l1, y_l1 = line_start
x_l2, y_l2 = line_end

for frame in all_frames:
    cur_ids = frames_id[frame].keys()  # ids текущего фрейма
    
    for cur_id in cur_ids:
        x_c, y_c = frames_id[frame][cur_id]  # текущие координаты боксов

        if not id_crossed[cur_id]:            
            if id_centers[cur_id]:  # центр бокса с предыдущего фрейма
                x_c_prev, y_c_prev = id_centers[cur_id]
                if check_cross(x_c, y_c, x_c_prev, y_c_prev, x_l1, y_l1, x_l2, y_l2):
                    crossings += 1
                    id_crossed[cur_id] = True
            else:
                id_centers[cur_id] = (x_c, y_c)

print(crossings)

15


In [8]:
# BEGIN STUDENT
def check_cross(x1, y1, x2, y2, x_l1, y_l1, x_l2, y_l2):
    """
    проверка, что отрезок (x1, y1) - (x2, y2) пересекает (x_l1, y_l1) - (x_l2, y_l2)
    """
    # проверяем что векторное произведение меняет знак z-компоненты
    z1 = (x2-x1)*(y_l1-y1) - (y2-y1)*(x_l1-x1)
    z2 = (x2-x1)*(y_l2-y1) - (y2-y1)*(x_l2-x1)

    z_l1 = (x_l2-x_l1)*(y1-y_l1) - (y_l2-y_l1)*(x1-x_l1)
    z_l2 = (x_l2-x_l1)*(y2-y_l2) - (y_l2-y_l1)*(x2-x_l2)

    return (z1*z2 < 0) and (z_l1*z_l2 < 0)
    

def count_line_crossings(tracks: list[dict], line_start: tuple[int, int], line_end: tuple[int, int]) -> int:
    """
    Подсчет количества пересечений линии объектами
    Args:
        tracks: список треков
        line_start: начальная точка линии (x, y)
        line_end: конечная точка линии (x, y)
    Returns:
        int: количество пересечений
    """
    frames_id = defaultdict(dict)
    all_frames = set()
    
    # проход по трекам, собираем историю
    for track in tracks:
        frame = track["frame"]
        all_frames.add(frame)
        id_frame = track["id"]
        x_center = track["bb_left"] + track["bb_width"]/2
        y_center = track["bb_top"] + track["bb_height"]/2
        frames_id[frame][id_frame] = (x_center, y_center)

    all_frames = sorted(list(all_frames))
    id_centers = defaultdict(bool)
    crossings = 0
    id_crossed = defaultdict(bool)
    
    x_l1, y_l1 = line_start
    x_l2, y_l2 = line_end
    
    for frame in all_frames:
        cur_ids = frames_id[frame].keys()  # ids текущего фрейма
        
        for cur_id in cur_ids:
            x_c, y_c = frames_id[frame][cur_id]  # текущие координаты боксов
    
            if not id_crossed[cur_id]:            
                if id_centers[cur_id]:  # центр бокса с предыдущего фрейма
                    x_c_prev, y_c_prev = id_centers[cur_id]
                    if check_cross(x_c, y_c, x_c_prev, y_c_prev, x_l1, y_l1, x_l2, y_l2):
                        crossings += 1
                        id_crossed[cur_id] = True
                else:
                    id_centers[cur_id] = (x_c, y_c)
    return crossings
# END STUDENT

In [9]:
# Загружаем результаты трекинга
with open("/kaggle/input/datasets/andreykurdyubov/cv-hw9-yolo-detections/tracking_results_2.json", 'r') as f:
    tracks = json.load(f)

# Определяем линию подсчета
# Для ADL-Rundle-3 определим вертикальную линию примерно в центре кадра
line_start = (960, 0)    # Верхняя точка линии (x=960, y=0)
line_end = (960, 1080)   # Нижняя точка линии (x=960, y=1080)

# Подсчитываем пересечения
crossings = count_line_crossings(tracks, line_start, line_end)

# Выводим статистику
print(f"Статистика пересечений:")
print(f"Всего пересечений линии: {crossings}")
print(f"Количество уникальных треков: {len(set(t['id'] for t in tracks))}")
print(f"Общее количество детекций в треках: {len(tracks)}")

Статистика пересечений:
Всего пересечений линии: 15
Количество уникальных треков: 63
Общее количество детекций в треках: 7881


Код функции загрузите в LMS.

In [ ]:
# решение автора
def count_line_crossings(tracks: list[dict], line_start: tuple[int, int], line_end: tuple[int, int]) -> int:
    def line_intersection(p1: tuple[float, float], p2: tuple[float, float],
                         p3: tuple[float, float], p4: tuple[float, float]) -> bool:
        # Проверка пересечения двух отрезков
        x1, y1 = p1
        x2, y2 = p2
        x3, y3 = p3
        x4, y4 = p4
        
        denominator = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
        if denominator == 0:
            return False
            
        t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denominator
        u = -((x1 - x2) * (y1 - y3) - (y1 - y2) * (x1 - x3)) / denominator
        
        return 0 <= t <= 1 and 0 <= u <= 1
    
    # Группируем треки по ID
    tracks_by_id = {}
    for track in tracks:
        track_id = track['id']
        if track_id not in tracks_by_id:
            tracks_by_id[track_id] = []
        tracks_by_id[track_id].append(track)
    
    crossings = 0
    
    # Проверяем каждый трек на пересечение с линией
    for track_id, track_frames in tracks_by_id.items():
        # Сортируем кадры по номеру
        track_frames.sort(key=lambda x: x['frame'])
        
        # Проверяем каждую пару последовательных позиций
        for i in range(len(track_frames) - 1):
            curr_pos = (track_frames[i]['bb_left'] + track_frames[i]['bb_width']/2,
                       track_frames[i]['bb_top'] + track_frames[i]['bb_height']/2)
            next_pos = (track_frames[i+1]['bb_left'] + track_frames[i+1]['bb_width']/2,
                       track_frames[i+1]['bb_top'] + track_frames[i+1]['bb_height']/2)
            
            if line_intersection(curr_pos, next_pos, line_start, line_end):
                crossings += 1
                break  # Считаем только первое пересечение для каждого объекта
    
    return crossings

# Заключение

В данном домашнем задании мы рассмотрели полный пайплайн системы отслеживания объектов, состоящий из нескольких ключевых этапов:

1. Работа с данными:
   - Реализовали класс для загрузки и обработки датасета MOT15
   - Научились работать с последовательностями изображений и аннотациями

2. Детекция объектов:
   - Использовали предобученную модель YOLOv8 для обнаружения людей на фреймах

3. Трекинг объектов:
   - Применили алгоритм DeepSORT для отслеживания объектов

4. Визуализация и анализ:
   - Создали инструменты для визуализации результатов трекинга
   - Реализовали подсчет пересечений заданной линии

Важно отметить, что каждый этап пайплайна может быть дополнительно оптимизирован и настроен под конкретную задачу, что открывает широкие возможности для дальнейшего развития системы.

## 2026.08.05 Сборка видео

In [10]:
# chatgpt
import json
from collections import defaultdict

import cv2
import numpy as np


def segments_intersect(p1, p2, p3, p4):
    """Проверка пересечения двух отрезков."""

    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    x4, y4 = p4

    denominator = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)

    if abs(denominator) < 1e-9:
        return False

    t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denominator
    u = -((x1 - x2) * (y1 - y3) - (y1 - y2) * (x1 - x3)) / denominator

    return 0 <= t <= 1 and 0 <= u <= 1


def id_to_color(track_id):
    """Постоянный цвет для каждого ID."""
    np.random.seed(track_id)
    return tuple(int(c) for c in np.random.randint(0, 255, 3))


def visualize_tracking_video(
    dataset,
    tracking_json,
    output_video,
    line_start,
    line_end,
    fps=30,
):
    # -------------------------
    # Загружаем результаты
    # -------------------------
    with open(tracking_json, "r") as f:
        tracks = json.load(f)

    tracks_by_frame = defaultdict(list)

    for track in tracks:
        tracks_by_frame[track["frame"]].append(track)

    # -------------------------
    # Создаем VideoWriter
    # -------------------------
    first_image, _ = dataset[0]

    height, width = first_image.shape[:2]

    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    # -------------------------
    # Данные для подсчета
    # -------------------------
    previous_centers = {}
    crossed_ids = set()

    # -------------------------
    # Основной цикл
    # -------------------------
    for frame_idx in range(len(dataset)):

        image, _ = dataset[frame_idx]

        frame = image.copy()

        # линия подсчета
        cv2.line(
            frame,
            line_start,
            line_end,
            (0, 255, 255),
            3,
        )

        current_tracks = tracks_by_frame.get(frame_idx + 1, [])

        for track in current_tracks:

            track_id = track["id"]

            left = int(track["bb_left"])
            top = int(track["bb_top"])
            width_box = int(track["bb_width"])
            height_box = int(track["bb_height"])

            right = left + width_box
            bottom = top + height_box

            center = (
                left + width_box / 2,
                top + height_box / 2,
            )

            color = id_to_color(track_id)

            # bbox
            cv2.rectangle(
                frame,
                (left, top),
                (right, bottom),
                color,
                2,
            )

            # центр
            cv2.circle(
                frame,
                (int(center[0]), int(center[1])),
                4,
                color,
                -1,
            )

            # ID
            cv2.putText(
                frame,
                f"ID {track_id}",
                (left, top - 7),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2,
            )

            # Проверяем пересечение
            if track_id in previous_centers:

                if (
                    track_id not in crossed_ids
                    and segments_intersect(
                        previous_centers[track_id],
                        center,
                        line_start,
                        line_end,
                    )
                ):
                    crossed_ids.add(track_id)

            previous_centers[track_id] = center

        # Счетчик
        cv2.putText(
            frame,
            f"Crossings: {len(crossed_ids)}",
            (25, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.2,
            (0, 0, 255),
            3,
        )

        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    writer.release()

    print(f"Video saved to {output_video}")
    print(f"Total crossings: {len(crossed_ids)}")

In [12]:
dataset = MOT15Dataset(
    "/kaggle/input/datasets/mdraselsarker/mot15-challenge-dataset/MOT15/train",
    "ADL-Rundle-6",
)

visualize_tracking_video(
    dataset=dataset,
    tracking_json="/kaggle/input/datasets/andreykurdyubov/cv-hw9-yolo-detections/tracking_results_2.json",
    output_video="tracking_result.mp4",
    line_start=(960, 0),
    line_end=(960, 1080),
    fps=30,
)

Video saved to tracking_result.mp4
Total crossings: 15
